# ESM-2 150M — DIMER E2E protein sequence-classification adaptation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/esm2-protein-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/esm2-protein-pipeline/blob/main/tutorials/esm2_protein_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-facebook%2Fesm2__t30__150M__UR50D-ffcc4d?style=flat)](https://huggingface.co/facebook/esm2_t30_150M_UR50D) [![Upstream](https://img.shields.io/badge/Upstream-facebookresearch%2Fesm-181717?style=flat&logo=github&logoColor=white)](https://github.com/facebookresearch/esm) [![bioRxiv](https://img.shields.io/badge/bioRxiv-2022.07.20.500902-b31b1b.svg)](https://doi.org/10.1101/2022.07.20.500902)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** protein sequence embeddings and bounded sequence-classification fine-tuning on labelled protein sequences

**This notebook is standalone.** It carries the repository's package (3 modules under `src/esm2_protein_pipeline/`, at revision `2bbcf0b38fe8`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `a695f6045e2e32885fa60af20c13cb35398ce30c` (~595 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned ESM-2 150M snapshot (6 files, ~595 MB), generates the deterministic 96-sequence tutorial dataset in code (no download), validates it against the sequence-classification contract, splits it into stratified train/validation/test sets, computes mean-pooled sequence embeddings, measures a majority-class and a composition baseline on the test split, runs a bounded AdamW fine-tuning of the classification head and the last two encoder layers, evaluates accuracy, macro-F1 and AUROC on the held-out test split, classifies six freshly generated sequences, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify prediction parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled protein dataset as a CSV (`id,sequence,label` header), JSON array or JSONL file. It passes through the same validation, stratified split, baselines, adaptation, held-out evaluation, inference, artifact export and reload-parity cells as the synthetic sample. The expected schema, the alphabet and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

ESM-2 is a protein language model: a 30-layer transformer encoder with rotary position embeddings, pretrained by masked-token prediction on UniRef50 protein sequences (Lin et al., 2022). The pinned checkpoint `facebook/esm2_t30_150M_UR50D` reads one amino-acid residue per token and produces a 640-dimensional hidden state per residue. This tutorial uses it in two ways: as a **representation model** (mean-pooled residue states give one embedding per sequence) and as the **base of a sequence classifier** (`EsmForSequenceClassification` adds a newly initialised head on the `<cls>` position, and a bounded gradient fine-tuning adapts that head together with the last encoder layers). The tutorial dataset is synthetic and deliberately order-sensitive: every sequence carries the same number of strongly hydrophobic residues, but in class `segment` they form one contiguous 18–22-residue stretch and in class `scattered` they are spread out. A residue-counting baseline cannot separate the classes; a model that reads the sequence can — so the comparison shows what fine-tuning adds.

**Learning objectives:** install the pinned runtime; inspect the carried pipeline, dataset and metrics modules; stage and digest-verify the immutable ESM-2 snapshot; validate a labelled protein dataset against explicit ceilings and split it without leakage; extract and inspect mean-pooled sequence embeddings; measure majority-class and composition baselines; run a bounded fine-tuning with explicit hyperparameters and a recorded trainable-parameter set; evaluate accuracy, macro-F1 and AUROC on an independent test split; classify new sequences with argmax scores; and export a safetensors adapter that reloads against the pinned base with verified prediction parity.

**This notebook does not demonstrate:** residue-level (token) classification, contact or structure prediction, masked-token scoring, full-parameter fine-tuning of all 30 layers, or arbitrary unvalidated dataset formats. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU in about a minute of model time on a workstation; CUDA is used automatically when present and shortens adaptation.
- **Knowledge:** amino-acid one-letter codes, what a train/validation/test split protects against, and how accuracy, macro-F1 and AUROC differ.
- **Data contract:** records are `{id, sequence, label}`; sequences are uppercase one-letter strings over `ACDEFGHIKLMNPQRSTVWY` plus the ambiguity codes `BUZOX`, at most 1,022 residues (`MAX_RESIDUES`), unique ids and unique sequences; at least 12 records and 3 per class, 2..20 classes. BYOD accepts CSV (`id,sequence,label`), JSON array or JSONL. Do not upload confidential or restricted data to a hosted runtime unless authorized.
- **Expected log lines:** loading `EsmForSequenceClassification` prints that the classifier head weights are newly initialised — that is the head this tutorial trains, not a defect.
- **External access:** the Hugging Face Hub only, to fetch the pinned `facebook/esm2_t30_150M_UR50D` snapshot (~595 MB in total) at revision `a695f6045e2e…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `safetensors` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'esm2-protein-pipeline',
    'repository_revision': '2bbcf0b38fe86ee4c348f177e557f713d0f6e374',
    'embedded_module': 'src/esm2_protein_pipeline/pipeline.py',
    'embedded_modules': ['src/esm2_protein_pipeline/pipeline.py', 'src/esm2_protein_pipeline/samples.py', 'src/esm2_protein_pipeline/metrics.py'],
    'module_sha256': '460e27e21658f5f35be21963977c19eea0ac502628326602c00c9ac9f925d676',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, safetensors
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'safetensors': safetensors.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/esm2_protein_pipeline/` @ `2bbcf0b38fe8`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/esm2_protein_pipeline/pipeline.py`

In [ ]:
"""ESM-2 150M (`facebook/esm2_t30_150M_UR50D`) DIMER pipeline: verified snapshot, residue-level
embeddings, and bounded sequence-classification fine-tuning with a portable adapter artifact.

Everything model-related is imported lazily so that snapshot verification and input validation run
(and can refuse) before `torch` or `transformers` are imported (fleet RTM-001).
"""

from __future__ import annotations

import hashlib
import json
import warnings
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "facebook/esm2_t30_150M_UR50D"
MODEL_REVISION = "a695f6045e2e32885fa60af20c13cb35398ce30c"
MODEL_LICENSE = "mit"
MODEL_KEY = "esm2-t30-150m-ur50d"
ARTIFACT_FORMAT = "org.valcorza.esm2-protein.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Ceilings. ESM-2 was trained on windows of 1,024 tokens; `<cls>` and `<eos>` take two of them, so
# 1,022 residues is the longest sequence that fits one training-length window. Rotary positions
# do not fail beyond that, they degrade silently, so the pipeline refuses longer sequences.
MAX_RESIDUES = 1_022
MAX_SEQUENCES_PER_CALL = 64
HIDDEN_SIZE = 640  # config.json hidden_size; the embedding width every `embed` row has
# The 20 standard amino acids plus the ambiguity/rare codes ESM-2's vocabulary carries. Anything
# else (lowercase, gaps, whitespace, digits) is a validation error, never a silent `<unk>`.
STANDARD_AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
EXTENDED_AMINO_ACIDS = "BUZOX"  # B=Asx, U=selenocysteine, Z=Glx, O=pyrrolysine, X=unknown
ALPHABET = frozenset(STANDARD_AMINO_ACIDS + EXTENDED_AMINO_ACIDS)


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the ESM-2 snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _verify_manifest(root, MODEL_ID, MODEL_REVISION)


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


INPUT_SCHEMA: dict[str, Any] = {
    "input": "1..MAX_SEQUENCES_PER_CALL protein sequences as uppercase one-letter strings",
    "sequences": [1, MAX_SEQUENCES_PER_CALL],
    "residues": [1, MAX_RESIDUES],
    "alphabet": "".join(sorted(ALPHABET)),
    "preprocessing": (
        "each sequence is tokenized one residue per token by the pinned EsmTokenizer, wrapped in "
        "<cls> ... <eos>, and right-padded to the longest sequence in the batch; embeddings are the "
        "mean of the last hidden state over residue tokens only (cls, eos and pad excluded)"
    ),
}


def _check_sequences(sequences: Any, names: Any = None) -> tuple[list[str], list[str]]:
    """Raise TypeError/ValueError naming the first violated ceiling; return (sequences, ids).

    ``embed``, ``classify`` and ``validate_inputs`` all route through this function so their
    acceptance criteria cannot diverge.
    """
    if isinstance(sequences, str | bytes) or not isinstance(sequences, Sequence):
        raise TypeError("sequences must be a list of str (one protein sequence per item)")
    if not 1 <= len(sequences) <= MAX_SEQUENCES_PER_CALL:
        raise ValueError(f"sequences must hold 1..{MAX_SEQUENCES_PER_CALL} items, got {len(sequences)}")
    checked: list[str] = []
    for i, seq in enumerate(sequences):
        if not isinstance(seq, str):
            raise TypeError(f"sequences[{i}] must be str, got {type(seq).__name__}")
        if not seq:
            raise ValueError(f"sequences[{i}] is empty")
        if len(seq) > MAX_RESIDUES:
            raise ValueError(f"sequences[{i}] has {len(seq)} residues; ceiling is {MAX_RESIDUES}")
        bad = sorted({ch for ch in seq if ch not in ALPHABET})
        if bad:
            raise ValueError(
                f"sequences[{i}] contains characters outside the amino-acid alphabet: {bad!r} "
                f"(allowed: uppercase {STANDARD_AMINO_ACIDS} plus {EXTENDED_AMINO_ACIDS}; strip gaps, "
                "whitespace and lowercase first)"
            )
        checked.append(seq)
    if names is None:
        ids = [f"seq-{i}" for i in range(len(checked))]
    else:
        if isinstance(names, str | bytes) or not isinstance(names, Sequence) or len(names) != len(checked):
            raise ValueError("names must be a list with exactly one id per sequence")
        ids = [str(n) for n in names]
        if len(set(ids)) != len(ids):
            raise ValueError("names must be unique")
    return checked, ids


def validate_inputs(sequences: Sequence[str], *, names: Sequence[str] | None = None) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, verdict).

    Rejection is reported by raising exactly as ``embed``/``classify`` would; a caller that wants
    the finding recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    checked, ids = _check_sequences(sequences, names)
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": sid,
                "residues": len(seq),
                "non_standard_residues": sum(ch in EXTENDED_AMINO_ACIDS for ch in seq),
            }
            for sid, seq in zip(ids, checked, strict=True)
        ],
        "n_sequences": len(checked),
        "max_residues_observed": max(len(seq) for seq in checked),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def _softmax(logits: Sequence[float]) -> list[float]:
    import math

    top = max(logits)
    exps = [math.exp(v - top) for v in logits]
    total = sum(exps)
    return [v / total for v in exps]


@dataclass
class ESM2Pipeline:
    """ESM-2 150M pipeline: `embed` (representations) always; `classify` after `adapt` or `from_artifact`.

    `_embedder(sequences)` returns one HIDDEN_SIZE-wide mean-pooled vector per sequence;
    `_classifier(sequences)` returns one logits row per sequence (None until adapted).
    """

    _embedder: Callable[[list[str]], list[list[float]]]
    device: str
    load_warnings: list[str] = field(default_factory=list)
    classes: list[str] = field(default_factory=list)
    _classifier: Callable[[list[str]], list[list[float]]] | None = None
    model: Any = None
    tokenizer: Any = None
    classifier_model: Any = None
    adaptation: dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ESM2Pipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(f"no snapshot manifest at {root} and allow_download={allow_download}")
        # Stage and verify before importing model libraries (RTM-001).
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        import torch
        from transformers import AutoTokenizer, EsmModel

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            tokenizer = AutoTokenizer.from_pretrained(
                str(root), local_files_only=True, trust_remote_code=False
            )
            model = EsmModel.from_pretrained(
                str(root),
                local_files_only=True,
                trust_remote_code=False,
                use_safetensors=True,
                add_pooling_layer=False,
            )
        model = model.to(resolved_device).eval()
        messages = [f"{w.category.__name__}: {w.message}" for w in caught]
        pipe = cls(cls._make_embedder(model, tokenizer, resolved_device), resolved_device, messages)
        pipe.model, pipe.tokenizer = model, tokenizer
        return pipe

    # -- backends ---------------------------------------------------------------------------------

    @staticmethod
    def _encode(tokenizer: Any, sequences: list[str], device: str) -> dict[str, Any]:
        batch = tokenizer(sequences, return_tensors="pt", padding=True, add_special_tokens=True)
        return {k: v.to(device) for k, v in batch.items()}

    @staticmethod
    def _residue_mask(batch: dict[str, Any], tokenizer: Any) -> Any:
        """1 for residue tokens, 0 for <cls>, <eos> and padding (the pooling window)."""
        ids = batch["input_ids"]
        mask = batch["attention_mask"].clone()
        for special in (tokenizer.cls_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id):
            if special is not None:
                mask = mask * (ids != special)
        return mask

    @classmethod
    def _make_embedder(
        cls, model: Any, tokenizer: Any, device: str
    ) -> Callable[[list[str]], list[list[float]]]:
        import torch

        def embedder(sequences: list[str]) -> list[list[float]]:
            batch = cls._encode(tokenizer, sequences, device)
            with torch.inference_mode():
                hidden = model(**batch).last_hidden_state
            mask = cls._residue_mask(batch, tokenizer).unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
            return pooled.float().cpu().tolist()

        return embedder

    @classmethod
    def _make_classifier(
        cls, model: Any, tokenizer: Any, device: str
    ) -> Callable[[list[str]], list[list[float]]]:
        import torch

        def classifier(sequences: list[str]) -> list[list[float]]:
            batch = cls._encode(tokenizer, sequences, device)
            # no_grad, not inference_mode: the ESM rotary layers cache cos/sin tables per sequence
            # length, and a cache built under inference_mode cannot be reused by the next training
            # epoch ("Inference tensors cannot be saved for backward").
            with torch.no_grad():
                logits = model(**batch).logits
            return logits.float().cpu().tolist()

        return classifier

    # -- public stages ----------------------------------------------------------------------------

    def embed(self, sequences: Sequence[str], *, names: Sequence[str] | None = None) -> dict[str, Any]:
        """Mean-pooled last-hidden-state representation per sequence (HIDDEN_SIZE floats each)."""
        checked, ids = _check_sequences(sequences, names)
        vectors = self._embedder(checked)
        if len(vectors) != len(checked) or any(len(v) != HIDDEN_SIZE for v in vectors):
            raise RuntimeError("backend returned embeddings of the wrong shape")
        return {
            "ids": ids,
            "embeddings": [[float(x) for x in v] for v in vectors],
            "dimension": HIDDEN_SIZE,
            "pooling": "mean of the last hidden state over residue tokens (cls/eos/pad excluded)",
            "unit": "one vector per whole sequence; representations, not predictions",
            "n_sequences": len(checked),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def classify(self, sequences: Sequence[str], *, names: Sequence[str] | None = None) -> dict[str, Any]:
        """Class scores and argmax label per sequence; requires a prior `adapt` or `from_artifact`."""
        if self._classifier is None or not self.classes:
            raise RuntimeError(
                "classify requires an adapted head: call adapt(...) or load from_artifact(...) first"
            )
        checked, ids = _check_sequences(sequences, names)
        logits = self._classifier(checked)
        predictions = []
        for sid, seq, row in zip(ids, checked, logits, strict=True):
            if len(row) != len(self.classes):
                raise RuntimeError("backend returned a logits row that does not match the class list")
            scores = _softmax(row)
            best = max(range(len(scores)), key=scores.__getitem__)
            predictions.append(
                {
                    "id": sid,
                    "residues": len(seq),
                    "label": self.classes[best],
                    "score": scores[best],
                    "scores": dict(zip(self.classes, scores, strict=True)),
                }
            )
        return {
            "predictions": predictions,
            "classes": list(self.classes),
            "decision_rule": (
                "argmax over softmax(logits); scores are softmax outputs, not calibrated probabilities"
            ),
            "n_sequences": len(checked),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "adaptation": dict(self.adaptation),
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Held-out classification metrics over labelled records (see metrics.classification_metrics)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        validate_dataset(records, classes=self.classes)
        sequences = [r["sequence"] for r in records]
        ids = [r["id"] for r in records]
        predicted: list[str] = []
        scores: list[list[float]] = []
        for start in range(0, len(sequences), MAX_SEQUENCES_PER_CALL):
            chunk = self.classify(
                sequences[start : start + MAX_SEQUENCES_PER_CALL],
                names=ids[start : start + MAX_SEQUENCES_PER_CALL],
            )
            for p in chunk["predictions"]:
                predicted.append(p["label"])
                scores.append([p["scores"][c] for c in self.classes])
        return classification_metrics([r["label"] for r in records], predicted, scores, self.classes)

    def adapt(
        self,
        train_records: Sequence[Mapping[str, Any]],
        val_records: Sequence[Mapping[str, Any]] | None = None,
        *,
        classes: Sequence[str] | None = None,
        epochs: int = 4,
        learning_rate: float = 1e-4,
        batch_size: int = 8,
        trainable_layers: int = 2,
        weight_decay: float = 0.01,
        seed: int = 42,
    ) -> dict[str, Any]:
        """Bounded gradient fine-tuning of a sequence-classification head on top of the verified base.

        Builds `EsmForSequenceClassification` from the pinned base weights (the classifier head is
        newly initialised), freezes every parameter except the head and the last `trainable_layers`
        encoder layers, and runs AdamW for `epochs` passes. Validation records are monitored per
        epoch only; the final epoch's weights are kept (no selection).
        """
        if self.model is None or self.tokenizer is None:
            raise RuntimeError("adapt requires a pipeline built by from_pretrained (no loaded base model)")
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not 1 <= int(epochs) <= 50:
            raise ValueError("epochs must be in 1..50 (tutorial-scale adaptation)")
        if not 1 <= int(batch_size) <= MAX_SEQUENCES_PER_CALL:
            raise ValueError(f"batch_size must be in 1..{MAX_SEQUENCES_PER_CALL}")
        if not 0 <= int(trainable_layers) <= 30:
            raise ValueError("trainable_layers must be in 0..30")
        train_manifest = validate_dataset(train_records, classes=classes)
        class_list = list(train_manifest["classes"])
        if val_records is not None:
            validate_dataset(val_records, classes=class_list)

        import random

        import torch
        from transformers import EsmForSequenceClassification

        random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        root = Path(getattr(self.model.config, "_name_or_path", DEFAULT_WEIGHTS_DIR))
        clf = EsmForSequenceClassification.from_pretrained(
            str(root),
            local_files_only=True,
            trust_remote_code=False,
            use_safetensors=True,
            num_labels=len(class_list),
        )
        clf = clf.to(self.device)
        for p in clf.parameters():
            p.requires_grad = False
        layers = clf.esm.encoder.layer
        for layer in layers[len(layers) - int(trainable_layers) :] if trainable_layers else []:
            for p in layer.parameters():
                p.requires_grad = True
        for p in clf.classifier.parameters():
            p.requires_grad = True
        if trainable_layers:
            for p in clf.esm.encoder.emb_layer_norm_after.parameters():
                p.requires_grad = True
        trainable = [n for n, p in clf.named_parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in clf.parameters() if p.requires_grad)
        n_total = sum(p.numel() for p in clf.parameters())
        optimizer = torch.optim.AdamW(
            [p for p in clf.parameters() if p.requires_grad], lr=learning_rate, weight_decay=weight_decay
        )
        label_index = {c: i for i, c in enumerate(class_list)}
        examples = [(r["sequence"], label_index[r["label"]]) for r in train_records]
        self.classes = class_list
        self.classifier_model = clf
        self._classifier = self._make_classifier(clf, self.tokenizer, self.device)

        history: list[dict[str, Any]] = []
        for epoch in range(1, int(epochs) + 1):
            clf.train()
            order = list(range(len(examples)))
            random.shuffle(order)
            total_loss, n_batches = 0.0, 0
            for start in range(0, len(order), int(batch_size)):
                batch_examples = [examples[i] for i in order[start : start + int(batch_size)]]
                batch = self._encode(self.tokenizer, [s for s, _ in batch_examples], self.device)
                labels = torch.tensor([y for _, y in batch_examples], device=self.device)
                optimizer.zero_grad()
                out = clf(**batch, labels=labels)
                out.loss.backward()
                optimizer.step()
                total_loss += float(out.loss.item())
                n_batches += 1
            clf.eval()
            entry: dict[str, Any] = {
                "epoch": epoch,
                "train_loss": round(total_loss / max(1, n_batches), 6),
                "n_batches": n_batches,
            }
            if val_records:
                val = self.evaluate(val_records)
                entry["val_accuracy"] = val["accuracy"]
                entry["val_macro_f1"] = val["macro_f1"]
            history.append(entry)
        clf.eval()
        self.adaptation = {
            "method": "gradient fine-tuning (AdamW) of the classification head"
            + (f" and the last {int(trainable_layers)} encoder layer(s)" if trainable_layers else ""),
            "classes": class_list,
            "epochs": int(epochs),
            "learning_rate": float(learning_rate),
            "batch_size": int(batch_size),
            "weight_decay": float(weight_decay),
            "trainable_layers": int(trainable_layers),
            "seed": int(seed),
            "precision": "float32",
            "trainable_parameters": int(n_trainable),
            "total_parameters": int(n_total),
            "trainable_parameter_names": trainable,
            "train_records": len(train_records),
            "val_records": len(val_records) if val_records else 0,
            "selection": "final epoch kept; validation metrics are monitoring only",
            "history": history,
        }
        return dict(self.adaptation)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Export the trainable tensors as safetensors plus a JSON manifest binding them to the base."""
        if self.classifier_model is None or not self.classes:
            raise RuntimeError("save_artifact requires an adapted head (call adapt first)")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adaptation.get("trainable_parameter_names", []))
        tensors = {
            k: v.detach().cpu().contiguous()
            for k, v in self.classifier_model.state_dict().items()
            if k in names
        }
        if not tensors:
            raise RuntimeError("no trainable tensors recorded; nothing to export")
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path))
        digest = hashlib.sha256(weights_path.read_bytes()).hexdigest()
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"model_id": MODEL_ID, "model_revision": MODEL_REVISION, "license": MODEL_LICENSE},
            "classes": list(self.classes),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": digest}
            ],
            "tensors": sorted(tensors),
            "adaptation": {k: v for k, v in self.adaptation.items() if k != "trainable_parameter_names"},
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Rebuild the classification head from an exported artifact (manifest verified before loading)."""
        if self.model is None or self.tokenizer is None:
            raise RuntimeError("load_artifact requires a pipeline built by from_pretrained")
        art = Path(artifact_dir)
        manifest_path = art / ARTIFACT_MANIFEST_NAME
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest not found: {manifest_path}")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("model_id"), base.get("model_revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError(f"artifact was trained on {base}, this package pins {MODEL_ID}@{MODEL_REVISION}")
        classes = [str(c) for c in manifest.get("classes", [])]
        if len(classes) < 2 or len(set(classes)) != len(classes):
            raise ValueError("artifact manifest must list at least two unique classes")
        for entry in manifest["files"]:
            fp = art / entry["path"]
            if not fp.is_file():
                raise FileNotFoundError(f"artifact file missing: {fp}")
            if fp.stat().st_size != entry["bytes"]:
                raise ValueError(f"{entry['path']}: size {fp.stat().st_size} != manifest {entry['bytes']}")
            if hashlib.sha256(fp.read_bytes()).hexdigest() != entry["sha256"]:
                raise ValueError(f"{entry['path']}: sha256 mismatch against the artifact manifest")
        from safetensors.torch import load_file
        from transformers import EsmForSequenceClassification

        root = Path(getattr(self.model.config, "_name_or_path", DEFAULT_WEIGHTS_DIR))
        clf = EsmForSequenceClassification.from_pretrained(
            str(root),
            local_files_only=True,
            trust_remote_code=False,
            use_safetensors=True,
            num_labels=len(classes),
        )
        tensors = load_file(str(art / ARTIFACT_WEIGHTS_NAME))
        expected = set(manifest.get("tensors", []))
        if set(tensors) != expected:
            raise ValueError("artifact tensors do not match the names listed in its manifest")
        missing, unexpected = clf.load_state_dict(tensors, strict=False)
        if unexpected:
            raise ValueError(
                f"artifact carries tensors the base architecture does not have: {sorted(unexpected)[:5]}"
            )
        clf = clf.to(self.device).eval()
        self.classes = classes
        self.classifier_model = clf
        self._classifier = self._make_classifier(clf, self.tokenizer, self.device)
        self.adaptation = {**manifest.get("adaptation", {}), "loaded_from_artifact": str(art)}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ESM2Pipeline:
        """Verified base snapshot + exported adapter, ready for `classify`."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 2/3:** `src/esm2_protein_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic in-code sample data and the labelled-dataset contract for ESM-2 sequence classification.

The tutorial task is synthetic and deliberately order-sensitive: every sequence carries the same
number of strongly hydrophobic residues, but in class `segment` they form one contiguous stretch
(18-22 residues, the length range of a membrane-spanning helix) and in class `scattered` they are
spread out so that no hydrophobic run is longer than 5. A model that only counts residues cannot
separate the classes; a model that reads the sequence can. This is sanity evidence for the
fine-tuning contract, not a biological benchmark (NOTEBOOK_SPEC 2.0 DAT8).
"""

from __future__ import annotations

import csv
import hashlib
import json
import random
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import ALPHABET, MAX_RESIDUES, STANDARD_AMINO_ACIDS` removed — names are kernel globals defined by the carried modules

DATASET_REPRESENTATION = "io.github.kurtvalcorza.dataset.protein.sequence-labels.v1"
SAMPLE_CLASSES: tuple[str, ...] = ("scattered", "segment")
SAMPLE_SEED = 20260918
SAMPLE_SIZE = 96  # 48 per class
MIN_RECORDS = 12
MAX_RECORDS = 5_000
MAX_CLASSES = 20
MIN_RECORDS_PER_CLASS = 3
MAX_ID_CHARS = 64
MAX_LABEL_CHARS = 64
REQUIRED_COLUMNS = ("id", "sequence", "label")

# Residues counted as strongly hydrophobic for the synthetic rule (Kyte-Doolittle > 1.8 plus W).
HYDROPHOBIC = "AILMFVW"
# Background composition (approximate UniProt frequencies, %); the generator samples from it.
_BACKGROUND = {
    "A": 8.3,
    "R": 5.5,
    "N": 4.1,
    "D": 5.5,
    "C": 1.4,
    "Q": 3.9,
    "E": 6.7,
    "G": 7.1,
    "H": 2.3,
    "I": 5.9,
    "L": 9.7,
    "K": 5.8,
    "M": 2.4,
    "F": 3.9,
    "P": 4.7,
    "S": 6.6,
    "T": 5.4,
    "W": 1.1,
    "Y": 2.9,
    "V": 6.9,
}
_POLAR = "".join(ch for ch in STANDARD_AMINO_ACIDS if ch not in HYDROPHOBIC)
_SEGMENT_LENGTH = (18, 22)
_SEQUENCE_LENGTH = (60, 120)
_MAX_SCATTERED_RUN = 5


def _draw_background(rng: random.Random, n: int) -> list[str]:
    letters = list(_BACKGROUND)
    weights = [_BACKGROUND[ch] for ch in letters]
    return rng.choices(letters, weights=weights, k=n)


def longest_hydrophobic_run(sequence: str) -> int:
    """Length of the longest contiguous run of residues in HYDROPHOBIC."""
    best = run = 0
    for ch in sequence:
        run = run + 1 if ch in HYDROPHOBIC else 0
        best = max(best, run)
    return best


def hydrophobic_fraction(sequence: str) -> float:
    return sum(ch in HYDROPHOBIC for ch in sequence) / len(sequence) if sequence else 0.0


def _make_segment(rng: random.Random) -> str:
    length = rng.randint(*_SEQUENCE_LENGTH)
    seg_len = rng.randint(*_SEGMENT_LENGTH)
    body = _draw_background(rng, length)
    # Polar background outside the segment keeps the hydrophobic count equal between classes.
    body = [ch if ch not in HYDROPHOBIC else rng.choice(_POLAR) for ch in body]
    start = rng.randint(2, length - seg_len - 2)
    segment = rng.choices(HYDROPHOBIC, k=seg_len)
    body[start : start + seg_len] = segment
    return "".join(body)


def _make_scattered(rng: random.Random, n_hydrophobic: int, length: int) -> str:
    for _ in range(1000):
        body = [rng.choice(_POLAR) for _ in range(length)]
        positions = rng.sample(range(length), n_hydrophobic)
        for pos in positions:
            body[pos] = rng.choice(HYDROPHOBIC)
        seq = "".join(body)
        if longest_hydrophobic_run(seq) <= _MAX_SCATTERED_RUN:
            return seq
    raise RuntimeError("could not place scattered hydrophobic residues without a long run")


def generate_sample_dataset(seed: int = SAMPLE_SEED, size: int = SAMPLE_SIZE) -> list[dict[str, Any]]:
    """`size` labelled records (half `segment`, half `scattered`), deterministic for a given seed.

    Each `scattered` record mirrors one `segment` record's length and hydrophobic residue count,
    so the two classes have identical composition statistics by construction.
    """
    if size < 2 or size % 2:
        raise ValueError("size must be an even number >= 2 (one scattered record per segment record)")
    rng = random.Random(seed)
    records: list[dict[str, Any]] = []
    for i in range(size // 2):
        seg = _make_segment(rng)
        n_h = sum(ch in HYDROPHOBIC for ch in seg)
        sca = _make_scattered(rng, n_h, len(seg))
        records.append({"id": f"seg-{i:03d}", "sequence": seg, "label": "segment"})
        records.append({"id": f"sca-{i:03d}", "sequence": sca, "label": "scattered"})
    return records


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """SHA-256 over the canonical (id, sequence, label) rows; recorded in provenance (OUT9)."""
    canon = json.dumps([[r["id"], r["sequence"], r["label"]] for r in records], separators=(",", ":"))
    return hashlib.sha256(canon.encode("utf-8")).hexdigest()


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    classes: Sequence[str] | None = None,
    min_records: int = MIN_RECORDS,
    min_per_class: int = MIN_RECORDS_PER_CLASS,
) -> dict[str, Any]:
    """Check a labelled dataset against the sequence-classification contract; return its manifest.

    Every error names the record and the violated rule (VAL4/DAT19). When `classes` is given the
    labels must be drawn from exactly that list (used for validation/test splits and BYOD inference
    against an adapted head); otherwise the sorted set of observed labels becomes the class list.
    """
    if isinstance(records, str | bytes | Mapping) or not isinstance(records, Sequence):
        raise TypeError("records must be a list of {'id', 'sequence', 'label'} mappings")
    if len(records) < min_records:
        raise ValueError(f"dataset has {len(records)} records; at least {min_records} are required")
    if len(records) > MAX_RECORDS:
        raise ValueError(f"dataset has {len(records)} records; ceiling is {MAX_RECORDS}")
    seen_ids: set[str] = set()
    seen_sequences: dict[str, str] = {}
    counts: dict[str, int] = {}
    lengths: list[int] = []
    for i, rec in enumerate(records):
        if not isinstance(rec, Mapping):
            raise TypeError(f"record[{i}] must be a mapping, got {type(rec).__name__}")
        missing = [c for c in REQUIRED_COLUMNS if c not in rec]
        if missing:
            raise ValueError(
                f"record[{i}] is missing required column(s) {missing}; required: {list(REQUIRED_COLUMNS)}"
            )
        rid = str(rec["id"]).strip()
        if not rid or len(rid) > MAX_ID_CHARS:
            raise ValueError(f"record[{i}] id must be 1..{MAX_ID_CHARS} characters")
        if rid in seen_ids:
            raise ValueError(f"record[{i}] duplicates id {rid!r}")
        seen_ids.add(rid)
        seq = rec["sequence"]
        if not isinstance(seq, str) or not seq:
            raise ValueError(f"record[{i}] ({rid}) sequence must be a non-empty string")
        if len(seq) > MAX_RESIDUES:
            raise ValueError(f"record[{i}] ({rid}) has {len(seq)} residues; ceiling is {MAX_RESIDUES}")
        bad = sorted({ch for ch in seq if ch not in ALPHABET})
        if bad:
            raise ValueError(
                f"record[{i}] ({rid}) sequence contains characters outside the amino-acid alphabet: {bad!r}"
            )
        if seq in seen_sequences:
            raise ValueError(f"record[{i}] ({rid}) duplicates the sequence of {seen_sequences[seq]!r}")
        seen_sequences[seq] = rid
        label = rec["label"]
        if not isinstance(label, str) or not label.strip() or len(label) > MAX_LABEL_CHARS:
            raise ValueError(
                f"record[{i}] ({rid}) label must be a non-empty string of at most {MAX_LABEL_CHARS} chars"
            )
        counts[label] = counts.get(label, 0) + 1
        lengths.append(len(seq))
    if classes is None:
        class_list = sorted(counts)
    else:
        class_list = [str(c) for c in classes]
        unknown = sorted(set(counts) - set(class_list))
        if unknown:
            raise ValueError(f"labels {unknown} are not in the class list {class_list}")
    if len(class_list) < 2:
        raise ValueError(f"classification needs at least 2 classes, found {class_list}")
    if len(class_list) > MAX_CLASSES:
        raise ValueError(f"{len(class_list)} classes exceeds the ceiling of {MAX_CLASSES}")
    thin = [c for c in class_list if counts.get(c, 0) < min_per_class]
    if thin:
        raise ValueError(f"classes {thin} have fewer than {min_per_class} records each (class coverage rule)")
    return {
        "verdict": "accepted",
        "representation": DATASET_REPRESENTATION,
        "n_records": len(records),
        "classes": class_list,
        "class_counts": {c: counts.get(c, 0) for c in class_list},
        "residues": {"min": min(lengths), "max": max(lengths), "mean": round(sum(lengths) / len(lengths), 1)},
        "ceilings": {
            "max_residues": MAX_RESIDUES,
            "max_records": MAX_RECORDS,
            "max_classes": MAX_CLASSES,
            "min_records": min_records,
            "min_records_per_class": min_per_class,
        },
        "digest": dataset_digest(records),
        "findings": [],
    }


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.2,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Stratified random train/validation/test split (assumes rows are independent, SPL3).

    Each class is shuffled with `seed` and cut in the given proportions, so every split keeps the
    class proportions and every class has at least one record per split when the data allow it.
    """
    if not (0.0 < val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("val_fraction and test_fraction must be in (0, 1) and sum to less than 1")
    manifest = validate_dataset(records)
    rng = random.Random(seed)
    by_class: dict[str, list[dict[str, Any]]] = {c: [] for c in manifest["classes"]}
    for rec in records:
        by_class[rec["label"]].append(dict(rec))
    out: dict[str, list[dict[str, Any]]] = {"train": [], "validation": [], "test": []}
    for cls in manifest["classes"]:
        rows = by_class[cls]
        rng.shuffle(rows)
        n_val = max(1, round(len(rows) * val_fraction))
        n_test = max(1, round(len(rows) * test_fraction))
        if len(rows) - n_val - n_test < 1:
            raise ValueError(f"class {cls!r} has {len(rows)} records; too few to leave one per split")
        out["validation"].extend(rows[:n_val])
        out["test"].extend(rows[n_val : n_val + n_test])
        out["train"].extend(rows[n_val + n_test :])
    for part in out.values():
        rng.shuffle(part)
    return out


def load_byod_dataset(source: str | Path) -> list[dict[str, Any]]:
    """Read a user-supplied CSV (`id,sequence,label` header), JSON array, or JSONL file into records.

    Sequences are stripped of surrounding whitespace only; nothing else is rewritten (VAL7).
    The records are then validated with `validate_dataset`, whose errors name the offending row.
    """
    path = Path(source)
    if not path.is_file():
        raise FileNotFoundError(f"BYOD dataset file not found: {path}")
    text = path.read_text(encoding="utf-8-sig")
    if not text.strip():
        raise ValueError(f"BYOD dataset file is empty: {path}")
    suffix = path.suffix.lower()
    records: list[dict[str, Any]] = []
    if suffix == ".csv":
        reader = csv.DictReader(text.splitlines())
        header = [h.strip() for h in (reader.fieldnames or [])]
        missing = [c for c in REQUIRED_COLUMNS if c not in header]
        if missing:
            raise ValueError(f"CSV header {header} is missing required column(s) {missing}")
        for row in reader:
            records.append({c: (row.get(c) or "").strip() for c in REQUIRED_COLUMNS})
    elif suffix == ".jsonl":
        for line_no, line in enumerate(text.splitlines(), start=1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"line {line_no} is not valid JSON: {exc}") from exc
            records.append(item)
    elif suffix == ".json":
        try:
            data = json.loads(text)
        except json.JSONDecodeError as exc:
            raise ValueError(f"file is not valid JSON: {exc}") from exc
        if not isinstance(data, list):
            raise TypeError("JSON dataset must be a top-level array of objects")
        records = data
    else:
        raise ValueError(f"unsupported BYOD file type {suffix!r}; use .csv, .json or .jsonl")
    for rec in records:
        if isinstance(rec, Mapping) and isinstance(rec.get("sequence"), str):
            rec["sequence"] = rec["sequence"].strip()
    validate_dataset(records)
    return [dict(r) for r in records]


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write records as the BYOD CSV shape (`id,sequence,label`), so users have a template."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(REQUIRED_COLUMNS)
        for r in records:
            writer.writerow([r["id"], r["sequence"], r["label"]])
    return out

**Module 3/3:** `src/esm2_protein_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Classification metrics and trivial baselines for ESM-2 sequence-classification adaptation.

Pure Python (no scikit-learn): accuracy, macro-F1, per-class precision/recall/F1/support, and AUROC
(binary: positive class = the last entry of `classes`; multiclass: macro one-vs-rest), computed by
the Mann-Whitney rank statistic with average ranks for ties.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .samples import hydrophobic_fraction` removed — names are kernel globals defined by the carried modules


def _prf(hits: int, n_pred: int, n_true: int) -> dict[str, float]:
    precision = hits / n_pred if n_pred else 0.0
    recall = hits / n_true if n_true else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}


def auroc(y_true: Sequence[int], scores: Sequence[float]) -> float | None:
    """Area under the ROC curve for binary 0/1 labels; None when only one class is present."""
    n_pos = sum(1 for y in y_true if y == 1)
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    order = sorted(range(len(scores)), key=lambda i: scores[i])
    ranks = [0.0] * len(scores)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and scores[order[j + 1]] == scores[order[i]]:
            j += 1
        avg = (i + j + 2) / 2.0  # 1-based average rank of the tie block
        for k in range(i, j + 1):
            ranks[order[k]] = avg
        i = j + 1
    rank_sum = sum(r for r, y in zip(ranks, y_true, strict=True) if y == 1)
    return round((rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg), 4)


def classification_metrics(
    y_true: Sequence[str],
    y_pred: Sequence[str],
    scores: Sequence[Sequence[float]] | None,
    classes: Sequence[str],
) -> dict[str, Any]:
    """Discrete and ranking metrics over one evaluation split (labels are class names).

    `scores[i][k]` is the score of class `classes[k]` for record i (softmax outputs from the
    pipeline; any monotone score works for AUROC). Class order is preserved exactly as given.
    """
    if len(y_true) != len(y_pred):
        raise ValueError(f"{len(y_true)} labels vs {len(y_pred)} predictions")
    class_list = list(classes)
    unknown = sorted((set(y_true) | set(y_pred)) - set(class_list))
    if unknown:
        raise ValueError(f"labels outside the class list {class_list}: {unknown}")
    n = len(y_true)
    correct = sum(1 for t, p in zip(y_true, y_pred, strict=True) if t == p)
    per_class: dict[str, dict[str, Any]] = {}
    f1s: list[float] = []
    for c in class_list:
        hits = sum(1 for t, p in zip(y_true, y_pred, strict=True) if t == c and p == c)
        n_pred = sum(1 for p in y_pred if p == c)
        n_true = sum(1 for t in y_true if t == c)
        prf = _prf(hits, n_pred, n_true)
        per_class[c] = {**prf, "support": n_true, "predicted": n_pred}
        if n_true:
            f1s.append(prf["f1"])
    result: dict[str, Any] = {
        "n": n,
        "accuracy": round(correct / n, 4) if n else 0.0,
        "macro_f1": round(sum(f1s) / len(f1s), 4) if f1s else 0.0,
        "per_class": per_class,
        "classes": class_list,
        "decision_rule": "argmax over class scores",
        "auroc": None,
        "auroc_definition": None,
    }
    if scores is not None and n:
        if len(scores) != n or any(len(row) != len(class_list) for row in scores):
            raise ValueError("scores must be one row per record with one column per class")
        if len(class_list) == 2:
            pos = class_list[-1]
            result["auroc"] = auroc([1 if t == pos else 0 for t in y_true], [row[-1] for row in scores])
            result["auroc_definition"] = f"binary AUROC with positive class {pos!r} (last class in the list)"
        else:
            values = []
            for k, c in enumerate(class_list):
                a = auroc([1 if t == c else 0 for t in y_true], [row[k] for row in scores])
                if a is not None:
                    values.append(a)
            result["auroc"] = round(sum(values) / len(values), 4) if values else None
            result["auroc_definition"] = "macro-averaged one-vs-rest AUROC over classes present in the split"
    return result


def majority_baseline(
    train_records: Sequence[Mapping[str, Any]],
    eval_records: Sequence[Mapping[str, Any]],
    classes: Sequence[str],
) -> dict[str, Any]:
    """Predict the most frequent training class for every evaluation record (EVAL11)."""
    counts: dict[str, int] = {}
    for r in train_records:
        counts[r["label"]] = counts.get(r["label"], 0) + 1
    majority = max(sorted(counts), key=counts.__getitem__)
    metrics = classification_metrics(
        [r["label"] for r in eval_records], [majority] * len(eval_records), None, classes
    )
    return {"baseline": "majority-class", "predicted_label": majority, **metrics}


def composition_baseline(
    train_records: Sequence[Mapping[str, Any]],
    eval_records: Sequence[Mapping[str, Any]],
    classes: Sequence[str],
) -> dict[str, Any]:
    """Threshold on the hydrophobic residue fraction, fitted on the training split (binary only).

    The threshold and the class direction are chosen to maximise training accuracy; the evaluation
    split is never touched during fitting (SPL8). On the synthetic sample the two classes share
    their composition by construction, so this baseline is expected to sit near chance -- which
    is the point: it shows the fine-tuned model reads order, not counts.
    """
    class_list = list(classes)
    if len(class_list) != 2:
        raise ValueError("composition_baseline is defined for binary tasks only")
    lo, hi = class_list
    train_x = [hydrophobic_fraction(r["sequence"]) for r in train_records]
    train_y = [r["label"] for r in train_records]
    candidates = sorted(set(train_x))
    best = (-1.0, 0.0, True)  # accuracy, threshold, high_is_hi
    for t in candidates:
        for high_is_hi in (True, False):
            pred = [(hi if (x >= t) == high_is_hi else lo) for x in train_x]
            acc = sum(p == y for p, y in zip(pred, train_y, strict=True)) / len(train_y)
            if acc > best[0]:
                best = (acc, t, high_is_hi)
    _, threshold, high_is_hi = best
    eval_x = [hydrophobic_fraction(r["sequence"]) for r in eval_records]
    eval_pred = [(hi if (x >= threshold) == high_is_hi else lo) for x in eval_x]
    # Score for AUROC: the fraction itself, oriented so that a larger score favours `hi`.
    scores = [[1.0 - x, x] if high_is_hi else [x, 1.0 - x] for x in eval_x]
    metrics = classification_metrics([r["label"] for r in eval_records], eval_pred, scores, class_list)
    return {
        "baseline": "hydrophobic-fraction threshold",
        "threshold": round(threshold, 4),
        "rule": f"predict {hi!r} when fraction {'>=' if high_is_hi else '<'} {threshold:.4f}",
        "train_accuracy": round(best[0], 4),
        **metrics,
    }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `6`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `a695f6045e2e…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `ESM2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "esm2-t30-150m-ur50d",
  "modelId": "facebook/esm2_t30_150M_UR50D",
  "revision": "a695f6045e2e32885fa60af20c13cb35398ce30c",
  "files": [
    {
      "path": "README.md",
      "bytes": 1705,
      "sha256": "462a2f24724e19c6be0efab926315c294a863c9a9770e2c8b3d859b2d81a07de"
    },
    {
      "path": "config.json",
      "bytes": 779,
      "sha256": "e512f68ec444d99477703a9806639ca83da3dbc19f6c5fe428d6e5b7460972dc"
    },
    {
      "path": "model.safetensors",
      "bytes": 595257706,
      "sha256": "c3f1da8aea53bddd32c246c86168c23b9fd72341fb9db9a94436f855f5053566"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 125,
      "sha256": "3aedcd4211c0d43aec4e607ff60a63255f3174ead795e997350f09a5f8cd9ee1"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 95,
      "sha256": "7e9161ecdb548ec45a41cbc6b24aa4476fdd418461f491c4207baa99419a29ad"
    },
    {
      "path": "vocab.txt",
      "bytes": 93,
      "sha256": "0b82cc0a7c7cf9e567b1e5892d793285b9fbae822c964ca48696f7db44598e03"
    }
  ],
  "totalBytes": 595260503
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = ESM2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Sample dataset, validation and split

The default dataset is generated in code with a fixed seed (`SAMPLE_SEED`): 48 `segment` and 48 `scattered` sequences of 60–120 residues, paired so that each `scattered` record has exactly the length and hydrophobic residue count of one `segment` record. `validate_dataset` checks the schema, the alphabet, the residue ceiling, duplicate ids/sequences and class coverage before any model runs, and returns a manifest with the class counts, the ceilings and a SHA-256 digest of the rows. `split_dataset` then shuffles **within each class** with `SEED` and cuts 20 % validation / 20 % test, so all three splits keep the class balance; random splitting assumes the records are independent, which holds for generated data and must be checked for real proteins (homologous sequences across splits leak).

Look for: 96 records, classes `['scattered', 'segment']`, splits 56/20/20, and a written `outputs/esm2_protein_sample_dataset.csv` — the exact file shape BYOD expects. To use your own data, set `USE_BYOD = True` and re-run from this cell.

In [ ]:
import json
import os
from pathlib import Path

USE_BYOD = False  # @param {type:"boolean"}
VAL_FRACTION = 0.2  # @param {type:"number"}
TEST_FRACTION = 0.2  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    data_source = 'BYOD (' + file_name + ')'
else:
    records = generate_sample_dataset()
    data_source = f'synthetic hydrophobic-segment dataset (seed {SAMPLE_SEED}, {SAMPLE_SIZE} records)'

dataset_manifest = validate_dataset(records)
CLASSES = dataset_manifest['classes']
splits = split_dataset(records, val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION, seed=SEED)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(records, 'outputs/esm2_protein_sample_dataset.csv')

print({'data_source': data_source, 'n_records': dataset_manifest['n_records'], 'classes': CLASSES, 'class_counts': dataset_manifest['class_counts']})
print({'residues': dataset_manifest['residues'], 'ceilings': dataset_manifest['ceilings'], 'digest': dataset_manifest['digest'][:16] + '...'})
print({'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)})
example = train_records[0]
print({'example_id': example['id'], 'label': example['label'], 'residues': len(example['sequence']), 'longest_hydrophobic_run': longest_hydrophobic_run(example['sequence']), 'hydrophobic_fraction': round(hydrophobic_fraction(example['sequence']), 3)})
print(example['sequence'])

## 5. Sequence embeddings (representation, not prediction)

`pipe.embed` runs the verified base encoder and returns one 640-dimensional vector per sequence: the mean of the last hidden state over residue tokens only (`<cls>`, `<eos>` and padding are excluded). Embeddings are representations — they carry no label and no metric of their own; a downstream labelled task is what gives them meaning (EVAL9). The cell embeds eight validation sequences, writes them with their ids to `outputs/esm2_protein_embeddings.csv` (OUT4), and prints the mean cosine similarity within and between classes as an inspection, not an evaluation: the pretrained model has never seen this synthetic rule, so do not expect a clean separation before adaptation.

In [ ]:
import csv
import math

embed_records = val_records[:8]
embedding_result = pipe.embed([r['sequence'] for r in embed_records], names=[r['id'] for r in embed_records])
vectors = embedding_result['embeddings']
print({'n_sequences': embedding_result['n_sequences'], 'dimension': embedding_result['dimension'], 'pooling': embedding_result['pooling']})

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

within, between = [], []
for i in range(len(embed_records)):
    for j in range(i + 1, len(embed_records)):
        sim = cosine(vectors[i], vectors[j])
        (within if embed_records[i]['label'] == embed_records[j]['label'] else between).append(sim)
print({'mean_cosine_within_class': round(sum(within) / len(within), 4) if within else None, 'mean_cosine_between_classes': round(sum(between) / len(between), 4) if between else None, 'note': 'inspection only; embeddings are unlabelled representations'})

with open('outputs/esm2_protein_embeddings.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'label'] + [f'dim_{k}' for k in range(embedding_result['dimension'])])
    for r, vec in zip(embed_records, vectors):
        writer.writerow([r['id'], r['label']] + [f'{x:.6f}' for x in vec])
print('wrote outputs/esm2_protein_embeddings.csv')

## 6. Baselines on the test split

Two trivial predictors set the floor before any training (EVAL10/EVAL11). `majority_baseline` predicts the most frequent training class for every test record — 0.5 accuracy on a balanced split. `composition_baseline` fits one threshold on the hydrophobic residue **fraction** using the training split only (SPL8) and applies it to the test split; because the two synthetic classes share their composition by construction, it should also land near chance. A fine-tuned model that clears both has learned something about residue *order*. Both baselines are reported with the same `classification_metrics` fields as the model, so the numbers are directly comparable.

In [ ]:
baseline_majority = majority_baseline(train_records, test_records, CLASSES)
print({k: baseline_majority[k] for k in ('baseline', 'predicted_label', 'accuracy', 'macro_f1')})
if len(CLASSES) == 2:
    baseline_composition = composition_baseline(train_records, test_records, CLASSES)
    print({k: baseline_composition[k] for k in ('baseline', 'rule', 'train_accuracy', 'accuracy', 'macro_f1', 'auroc')})
else:
    baseline_composition = None
    print('composition baseline is defined for binary tasks only; skipped for', len(CLASSES), 'classes')

## 7. Bounded fine-tuning

`pipe.adapt` builds `EsmForSequenceClassification` from the verified base weights (the head is newly initialised — the log line says so), freezes every parameter except the classification head, the final layer norm and the last `TRAINABLE_LAYERS` encoder layers, and runs AdamW with the hyperparameters below (FT4/FT6): these are tutorial values chosen for a one-minute CPU run, not production settings. Validation metrics are computed after each epoch for **monitoring only**; the final epoch's weights are kept, so no selection happens on the validation split (EVAL14). Training loss going down is optimisation evidence, not task-quality evidence (FT7) — Section 8 is where quality is measured. Look for the trainable/total parameter counts (about 10.3 M of 148 M with two layers) and validation accuracy rising above the 0.5 baseline within the first epochs.

In [ ]:
import time

EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_LAYERS = 2  # @param {type:"integer"}

started = time.perf_counter()
adapt_result = pipe.adapt(
    train_records,
    val_records,
    classes=CLASSES,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    trainable_layers=TRAINABLE_LAYERS,
    seed=SEED,
)
adapt_seconds = round(time.perf_counter() - started, 1)
print({'method': adapt_result['method'], 'trainable_parameters': adapt_result['trainable_parameters'], 'total_parameters': adapt_result['total_parameters'], 'precision': adapt_result['precision'], 'device': pipe.device, 'seconds': adapt_seconds})
for step in adapt_result['history']:
    print({k: step[k] for k in step})

## 8. Held-out evaluation

`pipe.evaluate` classifies every record of a split and reports `accuracy` (discrete correctness under the argmax rule), `macro_f1` (the unweighted mean of per-class F1, which exposes a model that ignores a minority class), per-class precision/recall/F1 with support, and `auroc` (ranking quality of the positive-class score, independent of the argmax threshold; for more than two classes it is the macro one-vs-rest average). The **test split** was never used for training or monitoring, so its numbers are the independent evidence (SPL6/SPL7); the validation split is shown alongside for comparison. These are tutorial metrics on a synthetic 20-record split (EVAL6): a single holdout, no dispersion estimate. The report, with both baselines and the deltas against them, is written to `outputs/esm2_protein_evaluation_report.json`.

In [ ]:
val_metrics = pipe.evaluate(val_records)
test_metrics = pipe.evaluate(test_records)
print({'split': 'validation', **{k: val_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
print({'split': 'test', **{k: test_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
for cls_name, row in test_metrics['per_class'].items():
    print({'class': cls_name, **row})

evaluation_report = {
    'task': 'protein sequence classification (bounded fine-tuning of ESM-2 150M)',
    'evidence': 'tutorial sample-sanity metrics on one stratified holdout; not a benchmark',
    'estimation': 'single train/validation/test split, seed ' + str(SEED) + ', no dispersion estimate',
    'data_source': data_source,
    'dataset_digest': dataset_manifest['digest'],
    'classes': CLASSES,
    'splits': {'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)},
    'baselines': {'majority': baseline_majority, 'composition': baseline_composition},
    'validation_metrics': val_metrics,
    'test_metrics': test_metrics,
    'delta_vs_majority': {k: round(test_metrics[k] - baseline_majority[k], 4) for k in ('accuracy', 'macro_f1')},
    'adaptation': {k: v for k, v in adapt_result.items() if k != 'trainable_parameter_names'},
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/esm2_protein_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
print({'delta_vs_majority': evaluation_report['delta_vs_majority'], 'report': 'outputs/esm2_protein_evaluation_report.json'})

## 9. Inference on new sequences

`pipe.classify` returns, per sequence, the argmax `label`, its `score` and the full `scores` dictionary in class order. The scores are softmax outputs of a head trained on a few dozen sequences — **not calibrated probabilities** (UNC2); the only decision rule is argmax (UNC3), and a deployment that must trade false positives against false negatives owns its own threshold. On the default path the new sequences are generated with a different seed, so their true labels are known and shown as a check; on the BYOD path the first six test-split records stand in as new data (INF2). Predictions are written to `outputs/esm2_protein_predictions.csv` with ids and per-class scores.

In [ ]:
if USE_BYOD:
    new_records = test_records[:6]
    new_source = 'first six BYOD test-split records'
else:
    new_records = generate_sample_dataset(seed=7, size=6)
    new_source = 'freshly generated sequences (seed 7)'
input_manifest = validate_inputs([r['sequence'] for r in new_records], names=[r['id'] for r in new_records])
print({'new_source': new_source, 'verdict': input_manifest['verdict'], 'n_sequences': input_manifest['n_sequences'], 'max_residues_observed': input_manifest['max_residues_observed']})
inference_result = pipe.classify([r['sequence'] for r in new_records], names=[r['id'] for r in new_records])
predictions = inference_result['predictions']
print({'decision_rule': inference_result['decision_rule']})
n_match = 0
for p, r in zip(predictions, new_records):
    n_match += p['label'] == r['label']
    print({'id': p['id'], 'predicted': p['label'], 'score': round(p['score'], 4), 'true_label': r['label']})
print({'matches': n_match, 'of': len(new_records), 'note': 'sanity check on generated labels, not an evaluation'})

with open('outputs/esm2_protein_predictions.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'residues', 'predicted_label', 'score'] + [f'score_{c}' for c in CLASSES])
    for p in predictions:
        writer.writerow([p['id'], p['residues'], p['label'], f"{p['score']:.6f}"] + [f"{p['scores'][c]:.6f}" for c in CLASSES])
print('wrote outputs/esm2_protein_predictions.csv')

## 10. Export the adapter and verify a fresh reload

`pipe.save_artifact` writes only the trained tensors (head, final layer norm and the unfrozen encoder layers, about 41 MB for two layers) as `adapter.safetensors` plus a `manifest.json` that records the artifact format, the exact base model id and revision the tensors belong to (ART4), the class order, the tensor names, the file size and SHA-256, and the adaptation configuration (OUT8). `ESM2Pipeline.from_artifact` then re-verifies the base snapshot, checks the artifact manifest and digests **before** deserialising, rebuilds the classifier and overlays the tensors — a fresh object from files, not the in-memory model (VER2). The cell compares its predictions on the same new sequences with the pre-export ones: labels must match exactly and scores within `1e-5` (VER4).

In [ ]:
artifact_dir = Path('outputs/esm2_protein_adapter')
pipe.save_artifact(artifact_dir, metadata={'data_source': data_source, 'dataset_digest': dataset_manifest['digest'], 'test_metrics': {k: test_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
with open(artifact_dir / ARTIFACT_MANIFEST_NAME, encoding='utf-8') as f:
    artifact_manifest = json.load(f)
print({'format': artifact_manifest['format'], 'base_model': artifact_manifest['base_model'], 'classes': artifact_manifest['classes'], 'n_tensors': len(artifact_manifest['tensors']), 'files': artifact_manifest['files']})

reloaded_pipe = ESM2Pipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR)
reloaded_result = reloaded_pipe.classify([r['sequence'] for r in new_records], names=[r['id'] for r in new_records])
max_score_diff = 0.0
for before, after in zip(predictions, reloaded_result['predictions']):
    assert before['id'] == after['id'] and before['label'] == after['label'], f'reload parity failure on {before["id"]}'
    max_score_diff = max(max_score_diff, abs(before['score'] - after['score']))
assert max_score_diff < 1e-5, f'reload score drift {max_score_diff}'
print({'reload_parity': 'PASS', 'labels_equal': True, 'max_abs_score_diff': max_score_diff, 'loaded_from': reloaded_pipe.adaptation.get('loaded_from_artifact')})

## 11. Result export and provenance

The last output, `outputs/esm2_protein_result.json`, gathers everything a reader needs to interpret the files above: the notebook source revision, the model id, immutable revision and licence, the dataset source and digest, the adaptation configuration, baseline and held-out metrics, the new-sequence predictions, the artifact manifest, the reload-parity result, and the runtime versions and device (OUT6/OUT7). No credential is involved anywhere in this notebook, so none can leak into it (OUT10).

In [ ]:
import platform

result_payload = {
    'task': 'protein sequence classification adaptation (ESM-2 150M)',
    'pipeline_class': 'ESM2Pipeline',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'notebook_source': NOTEBOOK_SOURCE,
    'data_source': data_source,
    'dataset_manifest': dataset_manifest,
    'embedding_summary': {'n_sequences': embedding_result['n_sequences'], 'dimension': embedding_result['dimension'], 'pooling': embedding_result['pooling']},
    'evaluation_report': evaluation_report,
    'inference': {'new_source': new_source, 'decision_rule': inference_result['decision_rule'], 'predictions': predictions},
    'artifact_format': ARTIFACT_FORMAT,
    'artifact_format_version': ARTIFACT_FORMAT_VERSION,
    'artifact_manifest': artifact_manifest,
    'reload_parity': {'labels_equal': True, 'max_abs_score_diff': max_score_diff},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'safetensors': safetensors.__version__,
        'device': pipe.device,
        'precision': 'float32',
    },
}
with open('outputs/esm2_protein_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

The fine-tuned head separates `segment` from `scattered` sequences that share their residue composition, which the composition baseline cannot do: the model is using residue order, exactly what a protein language model is pretrained to represent. That is the whole claim of this notebook. The test split has 20 synthetic records, the metrics come from one seeded holdout with no dispersion estimate, and the classes are defined by a generator rule rather than by biology — so a perfect score here says the adaptation contract works, not that ESM-2 predicts membrane segments, localisation, function or anything else about real proteins. On real data the same workflow needs homology-aware splits (random splitting of related sequences leaks), labels from experiments or curated databases, and enough records per class for the numbers to mean something. The softmax scores are uncalibrated; embeddings are representations that need a labelled downstream task before any quality can be stated.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model, validate the demonstrated dataset contract, execute bounded fine-tuning, evaluate against trivial baselines on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or biological validity of the classes.

**Optional experiments (do not affect the default path):** set `TRAINABLE_LAYERS = 0` to train the head alone and compare the test metrics with the two-layer run; lower `EPOCHS` to 1 to see an under-trained head whose AUROC may still be high while accuracy sits near 0.5 (ranking before thresholding); or bring a real labelled dataset through BYOD and read the composition baseline first — if it already scores well, your labels may be predictable from composition alone.

## References

- Repository README: https://github.com/kurtvalcorza/esm2-protein-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/esm2-protein-pipeline/blob/main/MODEL_CARD.md
- Upstream model: https://huggingface.co/facebook/esm2_t30_150M_UR50D
- Upstream code: https://github.com/facebookresearch/esm
- Lin, Z. et al. (2022). Language models of protein sequences at the scale of evolution enable accurate structure prediction. bioRxiv 2022.07.20.500902. https://doi.org/10.1101/2022.07.20.500902